In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  REAL-TIME INDUSTRIAL HAND SAFETY DETECTOR                          ║
# ║  YOLOv8 + OpenCV  |  IoU-based danger zone overlap  |  Colab-ready  ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
#  PROBLEM SOLVED
#  ─────────────────────────────────────────────────────────────────────
#  Previous code detected the body/torso as "HAND" and flagged the wrong
#  region.  This version:
#    1. Runs YOLOv8 'person' detection → crops the upper-arm region only
#    2. Computes IoU (Intersection-over-Union) between each candidate and
#       the DANGER_ZONE rectangle
#    3. ONLY shows a box and triggers an alert when IoU ≥ IOU_THRESHOLD
#    4. Ignores every detection that does NOT overlap the danger zone
#
#  USAGE
#  ─────────────────────────────────────────────────────────────────────
#  • Single image  →  set MODE = "image"  and IMAGE_PATH = "your_file.jpg"
#  • Webcam        →  set MODE = "webcam"
#  • Video file    →  set MODE = "video"  and VIDEO_PATH = "your_video.mp4"

# ── Install (silent, safe to re-run) ─────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "-q", "ultralytics", "opencv-python-headless"])

# ── Imports ───────────────────────────────────────────────────────────
import cv2
import numpy as np
import time
from ultralytics import YOLO
from google.colab import files
from google.colab.patches import cv2_imshow   # Colab-safe display


# ═════════════════════════════════════════════════════════════════════
#  SECTION 1 — CONFIGURATION  (edit only here)
# ═════════════════════════════════════════════════════════════════════

# ── Input mode ────────────────────────────────────────────────────────
MODE       = "image"          # "image" | "webcam" | "video"
IMAGE_PATH = "withobject.jpeg"


# ── Danger zone  [x_left, y_top, width, height]  pixels ──────────────
#    Tip: run with SHOW_COORDS = True below, click on the image to read
#    pixel coordinates from the Matplotlib toolbar, then fill in here.
DANGER_ZONE = [600, 240, 350, 220]# ← tune to your image/frame

# ── IoU / overlap thresholds ──────────────────────────────────────────
#
#  IOU_THRESHOLD  — minimum Intersection-over-Union to trigger DANGER.
#    0.01  = any tiny pixel touch triggers alert  (most sensitive)
#    0.10  = hand must visibly enter the zone      (recommended start)
#    0.30  = hand must be well inside the zone     (strict)
#    0.50  = hand must be mostly inside the zone   (very strict)
#
#  INTERSECTION_THRESHOLD  — alternative: raw pixel overlap area.
#    Use this instead of IoU when the hand box size varies a lot.
#    Set to 0 to use IoU only.
IOU_THRESHOLD           = 0.05    # primary gate
INTERSECTION_THRESHOLD  = 0       # secondary gate (0 = disabled)

# ── YOLOv8 model ──────────────────────────────────────────────────────
YOLO_MODEL     = "yolov8n.pt"   # nano = fastest; yolov8s.pt = better
CONF_THRESHOLD = 0.30           # detection confidence

# ── Person-crop strategy ──────────────────────────────────────────────
#  The hand is in the upper portion of the 'person' box.
#  UPPER_FRACTION keeps only the top N% of the person bbox.
#  This avoids the torso/legs being tagged as "HAND".
UPPER_FRACTION = 0.55           # keep top 55% of person box

# ── Visual settings ───────────────────────────────────────────────────
DANGER_COLOR   = (0,   0, 255)  # BGR red
SAFE_DZ_COLOR  = (0, 255,   0)  # BGR green (zone, no overlap)
FONT           = cv2.FONT_HERSHEY_SIMPLEX


# ═════════════════════════════════════════════════════════════════════
#  SECTION 2 — OVERLAP LOGIC  (pure functions, no side effects)
# ═════════════════════════════════════════════════════════════════════

def xywh_to_xyxy(box):
    """Convert [x, y, w, h] → [x1, y1, x2, y2]."""
    x, y, w, h = box
    return x, y, x + w, y + h


def compute_iou(boxA, boxB):
    """
    Compute IoU between two [x,y,w,h] boxes.

    IoU = intersection_area / union_area

    Returns a float in [0, 1].
    A value of 0 means no overlap at all.
    A value of 1 means the boxes are identical.
    """
    ax1, ay1, ax2, ay2 = xywh_to_xyxy(boxA)
    bx1, by1, bx2, by2 = xywh_to_xyxy(boxB)

    # Intersection rectangle
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    inter_w = max(0, ix2 - ix1)
    inter_h = max(0, iy2 - iy1)
    inter_area = inter_w * inter_h

    if inter_area == 0:
        return 0.0

    area_A = (ax2 - ax1) * (ay2 - ay1)
    area_B = (bx2 - bx1) * (by2 - by1)
    union_area = area_A + area_B - inter_area

    return inter_area / union_area if union_area > 0 else 0.0


def compute_intersection_area(boxA, boxB):
    """
    Raw pixel area of the intersection rectangle.
    Useful when you want 'any N pixels inside the zone' as the gate.
    """
    ax1, ay1, ax2, ay2 = xywh_to_xyxy(boxA)
    bx1, by1, bx2, by2 = xywh_to_xyxy(boxB)
    ix1 = max(ax1, bx1);  iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2);  iy2 = min(ay2, by2)
    return max(0, ix2 - ix1) * max(0, iy2 - iy1)


def is_in_danger(hand_box, danger_zone,
                 iou_thresh=IOU_THRESHOLD,
                 inter_thresh=INTERSECTION_THRESHOLD):
    """
    Returns (in_danger: bool, iou: float, inter_area: int).

    Overlap is confirmed when EITHER:
      • IoU             ≥ iou_thresh   (primary check)
      • intersection px ≥ inter_thresh (secondary check, if enabled)
    """
    iou        = compute_iou(hand_box, danger_zone)
    inter_area = compute_intersection_area(hand_box, danger_zone)

    danger = iou >= iou_thresh
    if inter_thresh > 0:
        danger = danger or (inter_area >= inter_thresh)

    return danger, iou, inter_area


# ═════════════════════════════════════════════════════════════════════
#  SECTION 3 — HAND CANDIDATE EXTRACTION
# ═════════════════════════════════════════════════════════════════════

def extract_arm_hand_boxes(detections, model_names,
                           img_h, upper_frac=UPPER_FRACTION):
    """
    Given YOLO detections, build a list of candidate hand-region boxes.

    Strategy
    ────────
    For every 'person' detection:
      • Keep only the TOP `upper_frac` of the bounding box.
        This removes the torso and legs from consideration.
      • The resulting strip covers shoulder → wrist, which is where
        the hand actually is when reaching toward a machine.

    Why not detect 'hand' directly?
      COCO-trained YOLOv8 has no 'hand' class.  The person-crop
      approach is the best zero-training workaround.

    Returns list of [x, y, w, h] in image coordinates.
    """
    candidates = []
    for det in detections:
        label = model_names[int(det.cls[0])]
        if label != "person":
            continue

        x1, y1, x2, y2 = [int(v) for v in det.xyxy[0].tolist()]
        w = x2 - x1
        h = y2 - y1

        # Keep only the upper portion (arm/hand region)
        arm_h = int(h * upper_frac)
        candidates.append([x1, y1, w, arm_h])

    return candidates


# ═════════════════════════════════════════════════════════════════════
#  SECTION 4 — FRAME ANNOTATION
# ═════════════════════════════════════════════════════════════════════

def draw_dashed_rect(frame, x, y, w, h, color, thickness=2, dash=14):
    """Draw a dashed rectangle (OpenCV has no native dashed line)."""
    pts = [(x, y), (x+w, y), (x+w, y+h), (x, y+h), (x, y)]
    for i in range(len(pts) - 1):
        x1, y1 = pts[i];  x2, y2 = pts[i+1]
        seg_len = max(abs(x2-x1), abs(y2-y1))
        steps   = max(1, seg_len // dash)
        for s in range(0, steps, 2):
            t1 = s / steps;    t2 = min(1.0, (s+1) / steps)
            p1 = (int(x1+(x2-x1)*t1), int(y1+(y2-y1)*t1))
            p2 = (int(x1+(x2-x1)*t2), int(y1+(y2-y1)*t2))
            cv2.line(frame, p1, p2, color, thickness)


def annotate_frame(frame, danger_zone,
                   danger_hands, safe_hands,
                   fps=None):
    """
    Draw annotations on the frame IN-PLACE.

    Rules
    ─────
    • Danger zone box   → red if any overlap, green if no overlap
    • Hand boxes        → ONLY drawn when overlapping the danger zone
    • Safe hand boxes   → NOT drawn (per requirements)
    • Status banner     → top of frame
    • FPS counter       → bottom-right (real-time mode only)
    """
    dz_x, dz_y, dz_w, dz_h = danger_zone
    in_danger = len(danger_hands) > 0

    zone_color = DANGER_COLOR if in_danger else SAFE_DZ_COLOR

    # ── Danger zone rectangle (dashed) ─────────────────────────────
    draw_dashed_rect(frame, dz_x, dz_y, dz_w, dz_h, zone_color, 2)
    cv2.putText(frame, "DANGER ZONE",
                (dz_x + 4, dz_y - 8),
                FONT, 0.55, zone_color, 2, cv2.LINE_AA)

    # ── Draw ONLY the hands that are inside the danger zone ─────────
    for box, iou, inter in danger_hands:
        hx, hy, hw, hh = box
        # Solid red hand box
        cv2.rectangle(frame,
                      (hx, hy), (hx+hw, hy+hh),
                      DANGER_COLOR, 3)
        label = f"HAND  IoU:{iou:.2f}"
        cv2.putText(frame, label,
                    (hx + 4, hy - 8),
                    FONT, 0.55, DANGER_COLOR, 2, cv2.LINE_AA)

    # ── Status banner ───────────────────────────────────────────────
    h_frame = frame.shape[0]
    w_frame = frame.shape[1]
    banner_h = 52
    overlay  = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w_frame, banner_h),
                  (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)

    if in_danger:
        status_txt = "HAND IN DANGER ZONE!"
        status_col = DANGER_COLOR
        # Pulsing red tint on the whole frame when in danger
        pulse = frame.copy()
        cv2.rectangle(pulse, (0,0), (w_frame, h_frame),
                      (0, 0, 80), -1)
        cv2.addWeighted(pulse, 0.18, frame, 0.82, 0, frame)
    else:
        status_txt = "SAFE"
        status_col = SAFE_DZ_COLOR

    cv2.putText(frame, status_txt,
                (12, 36),
                FONT, 1.1, status_col, 3, cv2.LINE_AA)

    # ── FPS counter (real-time mode) ────────────────────────────────
    if fps is not None:
        cv2.putText(frame, f"FPS: {fps:.1f}",
                    (w_frame - 120, h_frame - 12),
                    FONT, 0.55, (180, 180, 180), 1, cv2.LINE_AA)

    return frame


# ═════════════════════════════════════════════════════════════════════
#  SECTION 5 — SINGLE-FRAME PROCESSING  (core pipeline, reused by all modes)
# ═════════════════════════════════════════════════════════════════════

def process_frame(frame, model, danger_zone):
    """
    Full pipeline for one frame.

    1. Run YOLOv8 inference
    2. Extract arm/hand candidate boxes
    3. For each candidate: compute IoU with danger zone
    4. Split into danger_hands vs ignored (safe)
    5. Return annotated frame + alert flag
    """
    img_h = frame.shape[0]

    # ── YOLOv8 inference ────────────────────────────────────────────
    results = model(frame, conf=CONF_THRESHOLD,
                    classes=[0],      # 0 = person only
                    verbose=False)

    all_dets = results[0].boxes if results else []

    # ── Extract arm-region candidates ───────────────────────────────
    candidates = extract_arm_hand_boxes(
        all_dets, model.names, img_h, UPPER_FRACTION)

    # ── Overlap test for every candidate ────────────────────────────
    danger_hands = []   # list of (box, iou, inter_area) — SHOWN
    safe_hands   = []   # list of box — NOT shown (per requirements)

    for box in candidates:
        in_danger_flag, iou, inter = is_in_danger(box, danger_zone)
        if in_danger_flag:
            danger_hands.append((box, iou, inter))
        else:
            safe_hands.append(box)

    # ── Annotate frame ───────────────────────────────────────────────
    annotated = annotate_frame(
        frame.copy(), danger_zone,
        danger_hands, safe_hands)

    alert = len(danger_hands) > 0
    return annotated, alert, danger_hands


# ═════════════════════════════════════════════════════════════════════
#  SECTION 6 — RUN MODES
# ═════════════════════════════════════════════════════════════════════

def run_image(model, danger_zone, path):
    """Process a single static image."""
    # Try to read directly; if not found, prompt upload
    frame = cv2.imread(path)
    if frame is None:
        print(f"'{path}' not found locally — opening upload dialog …")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")
        path  = list(uploaded.keys())[0]
        frame = cv2.imread(path)
    if frame is None:
        raise ValueError(f"Cannot read '{path}'.")

    print(f"Image loaded: {path}  ({frame.shape[1]}×{frame.shape[0]} px)")
    out, alert, hits = process_frame(frame, model, danger_zone)

    print(f"\nResult : {'⚠  HAND IN DANGER ZONE!' if alert else '✅  SAFE'}")
    for i, (box, iou, inter) in enumerate(hits, 1):
        print(f"  Hand {i}: IoU={iou:.3f}  "
              f"intersection={inter} px²  box={box}")

    cv2_imshow(out)
    cv2.imwrite("detection_result.jpg", out)
    print("Saved → detection_result.jpg")


def run_webcam(model, danger_zone):
    """
    Real-time webcam loop.
    Press  Q  to quit.
    (Works in local Python; in Colab use a video file instead.)
    """
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Cannot open webcam. "
                           "In Colab, set MODE='video' with a file.")
    prev = time.time()
    print("Webcam running … press Q to quit.")
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        out, alert, _ = process_frame(frame, model, danger_zone)
        now = time.time()
        fps = 1.0 / max(1e-6, now - prev);  prev = now
        # re-stamp FPS onto already-annotated frame
        cv2.putText(out, f"FPS: {fps:.1f}",
                    (out.shape[1]-120, out.shape[0]-12),
                    FONT, 0.55, (180,180,180), 1, cv2.LINE_AA)
        cv2.imshow("Safety Monitor", out)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()


def run_video(model, danger_zone, path):
    """
    Process a video file frame-by-frame.
    Output saved as output_safety.mp4.
    """
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: '{path}'")

    fps_in = cap.get(cv2.CAP_PROP_FPS) or 25
    w      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out_path = "output_safety.mp4"
    fourcc   = cv2.VideoWriter_fourcc(*"mp4v")
    writer   = cv2.VideoWriter(out_path, fourcc, fps_in, (w, h))

    print(f"Processing {total} frames at {fps_in:.1f} FPS …")
    danger_frames = 0
    prev = time.time()

    for idx in range(total):
        ok, frame = cap.read()
        if not ok:
            break
        out, alert, _ = process_frame(frame, model, danger_zone)
        if alert:
            danger_frames += 1
        writer.write(out)
        if idx % 30 == 0:
            now = time.time()
            proc_fps = 30 / max(1e-6, now - prev);  prev = now
            print(f"  Frame {idx}/{total}  proc_fps={proc_fps:.1f}  "
                  f"alerts so far={danger_frames}")

    cap.release()
    writer.release()
    print(f"\nDone. {danger_frames}/{total} danger frames.")
    print(f"Saved → {out_path}")


# ═════════════════════════════════════════════════════════════════════
#  SECTION 7 — ENTRY POINT
# ═════════════════════════════════════════════════════════════════════

print("=" * 58)
print("  INDUSTRIAL HAND SAFETY DETECTOR  |  YOLOv8 + IoU")
print("=" * 58)
print(f"  Mode         : {MODE}")
print(f"  Danger zone  : {DANGER_ZONE}")
print(f"  IoU threshold: {IOU_THRESHOLD}  "
      f"(lower = more sensitive)")
print(f"  YOLO model   : {YOLO_MODEL}")
print("=" * 58)

# Load model once — shared across all frames
print("\nLoading YOLOv8 model …")
model = YOLO(YOLO_MODEL)
print("Model ready.\n")

if MODE == "image":
    run_image(model, DANGER_ZONE, IMAGE_PATH)
elif MODE == "webcam":
    run_webcam(model, DANGER_ZONE)
elif MODE == "video":
    run_video(model, DANGER_ZONE, VIDEO_PATH)
else:
    raise ValueError(f"Unknown MODE '{MODE}'. Use 'image', 'webcam', or 'video'.")